In [0]:
# COMMAND ----------
# MAGIC %md
# MAGIC # Schema Enforcement & Schema Evolution in Delta Lake
# MAGIC
# MAGIC Two opposite behaviors, both controlled by Delta:
# MAGIC
# MAGIC | | What it does | Default? |
# MAGIC |---|---|---|
# MAGIC | **Schema Enforcement** | REJECTS writes that don't match the table schema | ✅ ON always |
# MAGIC | **Schema Evolution** | ACCEPTS writes with new/changed columns, updates schema | ❌ OFF, opt-in |
# MAGIC
# MAGIC This is a Delta Lake superpower — Parquet and CSV have neither.

In [0]:


from pyspark.sql.types import StructType, StructField, IntegerType, StringType, LongType

# Original schema: id, name, salary
employees = spark.createDataFrame([
    (1, "Aditi",  80000),
    (2, "Rahul",  90000),
    (3, "Priya",  75000),
], ["id", "name", "salary"])

employees.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("delta_catalog.delta_demo.employees")

print("Table created. Schema:")
spark.table("delta_catalog.delta_demo.employees").printSchema()

In [0]:
# COMMAND ----------
# MAGIC %md ## 2. Schema Enforcement — Delta REJECTS mismatched writes
# MAGIC
# MAGIC This is ON by default. You cannot turn it off.

# New data with an EXTRA column "department"
new_data_extra_col = spark.createDataFrame([
    (4, "Vikram", 85000, "Engineering"),
    (5, "Sneha",  70000, "Marketing"),
], ["id", "name", "salary", "department"])

# This will FAIL — Delta enforces the existing schema
try:
    new_data_extra_col.write.format("delta") \
        .mode("append") \
        .saveAsTable("delta_catalog.delta_demo.employees")
except Exception as e:
    print("ENFORCEMENT KICKED IN:")
    print(str(e)[:300])  # first 300 chars of error

In [0]:
# COMMAND ----------
# MAGIC %md ## 3. Schema Enforcement — WRONG data type

# Same columns but salary as STRING instead of LONG
wrong_type = spark.createDataFrame([
    (6, "Amit", "95000"),   # salary is string here
], ["id", "name", "salary"])

try:
    wrong_type.write.format("delta") \
        .mode("append") \
        .saveAsTable("delta_catalog.delta_demo.employees")
except Exception as e:
    print("TYPE MISMATCH CAUGHT:")
    print(str(e)[:300])

In [0]:
# COMMAND ----------
# MAGIC %md ## 4. Schema Enforcement — MISSING column

# Data missing the "salary" column entirely
missing_col = spark.createDataFrame([
    (7, "Neha"),
], ["id", "name"])

try:
    missing_col.write.format("delta") \
        .mode("append") \
        .saveAsTable("delta_catalog.delta_demo.employees")
except Exception as e:
    print("MISSING COLUMN CAUGHT:")
    print(str(e)[:300])

    #Note: Delta actually handles missing nullable columns by filling null — so this one might NOT fail. Try it and see. That's an interview gotcha.


In [0]:
# Neha's row was written with salary = null automatically:
# +---+------+------+
# | id|  name|salary|
# +---+------+------+
# |  1| Aditi| 80000|
# |  2| Rahul| 90000|
# |  3| Priya| 75000|
# |  7|  Neha|  null|  ← written successfully, salary filled as null
# +---+------+------+

# spark.table("delta_catalog.delta_demo.employees").show()
# Delta matches columns by name, not position. When incoming data is missing a column, Delta checks:

# Is the missing column nullable? → fill null, no error ✅
# Is the missing column non-nullable (NOT NULL constraint)? → reject with error ❌

# Since salary was defined as nullable (default), Delta quietly filled null.

# The interview trap:
# Interviewer asks: "Does Delta schema enforcement reject writes with missing columns?"
# ❌ Wrong answer: "Yes, always."
# ✅ Right answer: "It depends. Missing nullable columns are filled with null and accepted. Missing non-nullable columns are rejected. Delta matches by column name, not position — unlike Parquet which matches by position."


In [0]:
# COMMAND ----------
# MAGIC %md ## 5. Schema Evolution — opt in with mergeSchema
# MAGIC
# MAGIC When you WANT to add new columns, tell Delta explicitly.

# New data with extra "department" column
new_data_extra_col.write.format("delta")\
    .mode("append")\
        .option("mergeSchema", "true")\
            .saveAsTable("delta_catalog.delta_demo.employees")

print("Write succeeded. New schema:")
spark.table("delta_catalog.delta_demo.employees").printSchema()
spark.table("delta_catalog.delta_demo.employees").show()

In [0]:
# COMMAND ----------
# MAGIC %md ## 6. overwriteSchema — replace schema completely
# MAGIC
# MAGIC `mergeSchema` adds columns. `overwriteSchema` replaces the whole schema.
# MAGIC Use with caution — this is destructive.

completely_different = spark.createDataFrame([
    (1, "Laptop",  75000),
    (2, "Phone",   45000),
], ["product_id", "product_name", "price"])

# This would FAIL without overwriteSchema
# (column names don't match at all)
completely_different.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("delta_catalog.delta_demo.employees")

print("Schema completely replaced:")
spark.table("delta_catalog.delta_demo.employees").printSchema()

In [0]:
# Cell 3 — Type Mismatch (salary as String vs Long):
# DELTA_FAILED_TO_MERGE_FIELDS: Failed to merge fields 'salary' and 'salary'
# Delta found two salary columns — one Long (table), one String (incoming data). It tried to merge them and failed because String → Long is a narrowing conversion. Delta won't risk data loss by silently casting.

# Cell 2 — Extra Column (department):
# DELTA_METADATA_MISMATCH: A schema mismatch detected...
# To enable schema migration, set .option("mergeSchema", "true")
# Delta detected a new column department that doesn't exist in the table. It rejected the write AND helpfully told you exactly how to fix it — mergeSchema=true. That's a well-designed error message.

# The three enforcement behaviors — now you've seen all of them live:
# ScenarioDelta behavior You saw
# Extra new column ❌ Rejected Cell 2 error
# Wrong data type ❌ Rejected Cell 3 error
# Missing nullable column ✅ Accepted, null filled Cell 4 success

# Now run Cell 5 (mergeSchema) — this is where enforcement switches to evolution. You'll see the department column get added to the table and old rows backfilled with null.
# Paste the schema output after it runs.

In [0]:
# COMMAND ----------
# MAGIC %md ## 7. Check schema history — Delta remembers everything

spark.sql("""
    DESCRIBE HISTORY delta_catalog.delta_demo.employees
""").select("version", "timestamp", "operation", "operationParameters").display()

In [0]:
# COMMAND ----------
# MAGIC %md ## 8. Interview Gotchas
# MAGIC
# MAGIC 1. **Schema enforcement is always ON** — no flag to disable it. It's a Delta guarantee.
# MAGIC 2. **`mergeSchema` adds columns, never removes** — safe for additive changes.
# MAGIC 3. **Old rows get null for new columns** — downstream queries must handle nulls.
# MAGIC 4. **`overwriteSchema` is destructive** — requires `mode("overwrite")`, not `append`.
# MAGIC 5. **Type widening** — Delta 3.x supports widening (int → long) via `ALTER TABLE`. Narrowing (long → int) is always rejected.
# MAGIC 6. **Missing nullable columns** → Delta fills null, no error. Missing non-nullable columns → error.
# MAGIC 7. **Column order doesn't matter** — Delta matches by name, not position. Parquet matches by position — classic trap question.
# MAGIC 8. **Spark SQL equivalent of mergeSchema:**
# MAGIC ```sql
# MAGIC SET spark.databricks.delta.schema.autoMerge.enabled = true;
# MAGIC INSERT INTO employees SELECT * FROM new_data;
# MAGIC ```

In [0]:
# COMMAND ----------
# MAGIC %md ## 9. Summary — Decision Tree
# MAGIC
# MAGIC ```
# MAGIC Writing new data to Delta table?
# MAGIC │
# MAGIC ├── Same schema as table?
# MAGIC │     └── Just write. Enforcement passes. ✅
# MAGIC │
# MAGIC ├── Extra NEW columns in incoming data?
# MAGIC │     └── Add option("mergeSchema", "true")
# MAGIC │         → Old rows get null for new columns
# MAGIC │
# MAGIC ├── Completely different schema?
# MAGIC │     └── Add option("overwriteSchema", "true") + mode("overwrite")
# MAGIC │         → Destructive. Old data gone.
# MAGIC │
# MAGIC └── Wrong types or missing required columns?
# MAGIC       └── Fix the data first. Delta will not accept it.
# MAGIC ```

# Cleanup
spark.sql("DROP TABLE IF EXISTS delta_catalog.delta_demo.employees")